In [ ]:
import sys, math, yaml
sys.path.insert(0, "/home2/s4636708/master_thesis_project/src")
from galaxy_sidm.mock.residuals import pv_residuals

MARTINI = "/scratch/s4636708/aida/derived/martini"
CALIBRATION = "/home2/s4636708/master_thesis_project/config/residual_calibration.yaml"
SNAP_Z = {17: 5, 21: 4, 25: 3, 33: 2, 50: 1, 67: 0.5}
RES = 1                                  # which PV residual to cut on: 1, 2 or 3
VERSIONS = ["raw", "floor subtracted"]   # pv_residuals returns (raw, floor subtracted)

In [ ]:
# threshold = midpoint between the highest PV residual of the calibration discs
# and the lowest of the calibration perturbed galaxies (unsure discs don't enter),
# for the first model and redshift in the calibration file, once per version
labels = yaml.safe_load(open(CALIBRATION))
MODEL = next(iter(labels))
ZKEY = next(iter(labels[MODEL]))
cal = labels[MODEL][ZKEY]
disc = [pv_residuals(f"{MARTINI}/{ZKEY}/{MODEL}/gal_{s:06d}") for s in cal["discs"]]
pert = [pv_residuals(f"{MARTINI}/{ZKEY}/{MODEL}/gal_{s:06d}") for s in cal["perturbed"]]

THRESHOLD = []
for v, name in enumerate(VERSIONS):
    highest_disc = max(r[v][RES - 1] for r in disc if not math.isnan(r[v][RES - 1]))
    lowest_perturbed = min(r[v][RES - 1] for r in pert if not math.isnan(r[v][RES - 1]))
    THRESHOLD.append((highest_disc + lowest_perturbed) / 2)
    print(f"res{RES} {name}, calibration {MODEL} {ZKEY}: highest disc {highest_disc:.4g}, "
          f"lowest perturbed {lowest_perturbed:.4g} -> threshold {THRESHOLD[v]:.4g}")
    if highest_disc >= lowest_perturbed:
        print("  warning: the calibration discs and perturbed galaxies overlap")

In [ ]:
counts = {}   # (model, z) -> [total, no fit, kept raw, kept floor subtracted]
for line in open(MARTINI + "/manifest.txt"):
    model, snap, sub = line.split()
    z = SNAP_Z[int(snap)]
    res = pv_residuals(f"{MARTINI}/z{z:g}/{model}/gal_{int(sub):06d}")
    c = counts.setdefault((model, z), [0, 0, 0, 0])
    c[0] += 1
    if math.isnan(res[0][0]):
        c[1] += 1
        continue
    for v in range(2):
        if res[v][RES - 1] < THRESHOLD[v]:   # a nan (no signal above the noise) is not kept
            c[2 + v] += 1

In [ ]:
# left out = fitted but not below the threshold
print(f"PV res{RES}              raw                floor subtracted")
print("model   z     total  kept  left out     kept  left out     no fit")
for (model, z), (total, nofit, *kept) in sorted(counts.items()):
    row = f"{model:6s}  {z:<4g}  {total:5d}"
    for k in kept:
        out = total - nofit - k
        row += f"  {k:4d}  {out:4d} ({100 * out / total:3.0f}%)"
    print(row + f"  {nofit:6d}")